# Unified RL-Assisted MPC Horizons

This notebook replaces the split nominal and disturbance horizon workflows with a single `RUN_MODE` switch at the top. It uses the shared reward, observer, horizon-runner, and plotting modules so the notebook stays thin while preserving the current horizon experiment behavior.

In [ ]:
from pathlib import Path
import os

RUN_MODE = "nominal"  # switch to "disturb" for disturbance mode


def resolve_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "Simulation").exists() and (candidate / "utils").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current notebook working directory.")


REPO_ROOT = resolve_repo_root(Path.cwd())
os.chdir(REPO_ROOT)

RUN_PROFILE_MAP = {
    "nominal": {
        "use_disturbance": False,
        "baseline_mpc_path": REPO_ROOT / "Data" / "mpc_results_nominal.pickle",
        "result_prefix": "horizon_nominal_unified",
        "compare_prefix": "nominal_compare_horizon_unified",
        "plot_start_episode": 5,
        "compare_start_episode": 1,
        "compare_mode": "nominal",
    },
    "disturb": {
        "use_disturbance": True,
        "baseline_mpc_path": REPO_ROOT / "Data" / "mpc_results_dist.pickle",
        "result_prefix": "horizon_disturb_unified",
        "compare_prefix": "disturb_compare_horizon_unified",
        "plot_start_episode": 3,
        "compare_start_episode": 2,
        "compare_mode": "disturb",
    },
}

if RUN_MODE not in RUN_PROFILE_MAP:
    raise ValueError(f"RUN_MODE must be one of {list(RUN_PROFILE_MAP)}")

RUN_PROFILE = RUN_PROFILE_MAP[RUN_MODE]
print(f"Repo root: {REPO_ROOT}")
print(f"Run mode : {RUN_MODE}")
print(f"Baseline : {RUN_PROFILE['baseline_mpc_path']}")

In [ ]:
import numpy as np
import torch

from DQN.dqn_agent import DQNAgent
from Simulation.system_functions import PolymerCSTR
from utils.helpers import apply_min_max, build_horizon_recipes, load_and_prepare_system_data
from utils.horizon_runner import run_dqn_mpc_horizon_supervisor
from utils.observer import compute_observer_gain
from utils.plotting import compare_mpc_rl_from_dirs, plot_horizon_results
from utils.rewards import make_reward_fn_relative_QR

In [ ]:
# First initiate the system
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

delta_t = 0.5
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr.ss_inputs, "y_ss": cstr.y_ss}

setpoint_y = np.array([[3.2, 321.0], [4.5, 325.0]])
u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
    data_dir=str(REPO_ROOT / "Data"),
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]

min_max_dict = dict(system_data["min_max_dict"])
min_max_dict["x_max"] = np.asarray(system_data["min_max_states"]["max_s"], float)
min_max_dict["x_min"] = np.asarray(system_data["min_max_states"]["min_s"], float)

inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324.0], [3.4, 321.0]])
y_sp_scenario = apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]
)

n_tests = 200
set_points_len = 400
TEST_CYCLE = [False, False, False, False, False]
warm_start = 5

In [ ]:
poles = np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966, 0.42900673, 0.4228262, 0.96916776, 0.91230187])
L = compute_observer_gain(A_aug, C_aug, poles)

PREDICT_GRID = list(range(8, 20))
CONTROL_GRID = list(range(3, 10))
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
STATE_DIM = int(A_aug.shape[0]) + int(C_aug.shape[0]) + int(B_aug.shape[1])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HIDDEN_LAYERS = [512, 512, 512, 512, 512]
BUFFER_SIZE = 40_000
DECISION_INTERVAL = 4

predict_h = 9
cont_h = 3
Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])
Q_diag = np.array([518.0, 90.0])
R_diag = np.array([90.0, 90.0])
reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    n_inputs=2,
    k_rel=k_rel,
    band_floor_phys=band_floor_phys,
    Q_diag=Q_diag,
    R_diag=R_diag,
    tau_frac=0.7,
    gamma_out=0.5,
    gamma_in=0.5,
    beta=7.0,
    gate="geom",
    lam_in=1.0,
    bonus_kind="exp",
    bonus_k=12.0,
    bonus_p=0.6,
    bonus_c=20.0,
)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.85
qs_change = 1.3
ha_change = 0.85

print(f"Using device: {DEVICE}")
print(f"Number of horizon actions: {len(HORIZON_RECIPES)}")
print(reward_params)

dqn_agent = DQNAgent(
    state_dim=STATE_DIM,
    action_dim=len(HORIZON_RECIPES),
    hidden_dim=HIDDEN_LAYERS,
    gamma=0.99,
    lr=1e-4,
    batch_size=128,
    buffer_size=BUFFER_SIZE,
    grad_clip_norm=10.0,
    double_dqn=True,
    target_update="soft",
    tau=0.01,
    hard_update_interval=10_000,
    target_combine="q1",
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    device=DEVICE,
    eps_start=0.3,
    eps_end=0.01,
    eps_decay_rate=0.99999,
    eps_decay_mode="exp",
)

In [ ]:
horizon_cfg = {
    "mode": RUN_MODE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "decision_interval": DECISION_INTERVAL,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "reward_scale": 0.01,
}

runtime_ctx = {
    "system": PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t),
    "y_sp_scenario": y_sp_scenario,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "agent": dqn_agent,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "L": L,
    "data_min": data_min,
    "data_max": data_max,
    "horizon_recipes": HORIZON_RECIPES,
    "reward_fn": reward_fn,
}

result_bundle = run_dqn_mpc_horizon_supervisor(horizon_cfg=horizon_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = RUN_PROFILE["baseline_mpc_path"]
result_bundle["reward_params"] = reward_params
result_bundle["run_profile"] = RUN_PROFILE

print(f"nFE: {result_bundle['nFE']}")
print(f"Avg reward samples: {len(result_bundle['avg_rewards'])}")

In [ ]:
out_dir_rl = plot_horizon_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": REPO_ROOT / "Data",
        "prefix_name": RUN_PROFILE["result_prefix"],
        "start_episode": RUN_PROFILE["plot_start_episode"],
        "recipe_counts": True,
        "save_pdf": False,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=RUN_PROFILE["baseline_mpc_path"],
    reward_fn=reward_fn,
    directory=REPO_ROOT / "Result",
    prefix_name=RUN_PROFILE["compare_prefix"],
    compare_mode=RUN_PROFILE["compare_mode"],
    start_episode=RUN_PROFILE["compare_start_episode"],
)

print(f"RL plots saved to      : {out_dir_rl}")
print(f"Comparison plots saved : {out_dir_cmp}")